# Il filtraggio collaborativo

Il codice del capitolo [«Il filtraggio collaborativo»](https://book.paithon.it/main/SistemiRaccomandazione/filtraggio-collaborativo.html) — *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro — [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q torch torchvision

## Il filtraggio collaborativo

[Leggi la pagina](https://book.paithon.it/main/SistemiRaccomandazione/filtraggio-collaborativo.html)


### Il modello in PyTorch


In [ ]:
import torchfrom torch import nnclass FattorizzazioneMatrici(nn.Module):    def __init__(self, n_utenti, n_film, k=32):        super().__init__()        self.P = nn.Embedding(n_utenti, k)    # fattori latenti degli utenti        self.Q = nn.Embedding(n_film, k)      # fattori latenti dei film        self.b_u = nn.Embedding(n_utenti, 1)  # bias di utente        self.b_i = nn.Embedding(n_film, 1)    # bias di film        self.mu = nn.Parameter(torch.tensor(3.0))  # media globale        nn.init.normal_(self.P.weight, std=0.05)   # si parte quasi dalla media        nn.init.normal_(self.Q.weight, std=0.05)        nn.init.zeros_(self.b_u.weight)        nn.init.zeros_(self.b_i.weight)    def forward(self, u, i):        interazione = (self.P(u) * self.Q(i)).sum(dim=1)  # prodotto scalare        return (self.mu + self.b_u(u).squeeze(1)                + self.b_i(i).squeeze(1) + interazione)

In [ ]:
from torch.utils.data import DataLoader, TensorDatasettorch.manual_seed(0)n_utenti, n_film, k_vero = 300, 200, 4P_vero = torch.randn(n_utenti, k_vero)   # gusti "veri", nascostiQ_vero = torch.randn(n_film, k_vero)     # tratti "veri", nascostin_voti = 6_000        # 6.000 voti su 60.000 celle: matrice piena al 10% circau = torch.randint(0, n_utenti, (n_voti,))i = torch.randint(0, n_film, (n_voti,))affinita = (P_vero[u] * Q_vero[i]).sum(1)voti = (3 + 1.2 * affinita / affinita.std()).clamp(1, 5)  # scala 1-5loader = DataLoader(TensorDataset(u, i, voti), batch_size=256, shuffle=True)

In [ ]:
modello = FattorizzazioneMatrici(n_utenti, n_film, k=8)ottim = torch.optim.Adam(modello.parameters(), lr=0.01, weight_decay=1e-4)criterio = nn.MSELoss()for epoca in range(30):    errore_tot = 0.0    for batch_u, batch_i, batch_r in loader:        pred = modello(batch_u, batch_i)        loss = criterio(pred, batch_r)      # MSE sui soli voti osservati        ottim.zero_grad()        loss.backward()        ottim.step()        errore_tot += loss.item() * len(batch_r)    if (epoca + 1) % 10 == 0:        print(f"epoca {epoca + 1:2d} · MSE {errore_tot / n_voti:.3f}")

## La raccomandazione neurale

[Leggi la pagina](https://book.paithon.it/main/SistemiRaccomandazione/raccomandazione-neurale.html)


### Imparare a ordinare: BPR


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

```
